<a href="https://colab.research.google.com/github/abdel-2000/pipeline-arabo/blob/main/pipeline_arabo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("LLM Arabic Project")


LLM Arabic Project


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [12]:
# ============================================================
# CELLA 1 — Installa tutto
# ============================================================
!pip install pdfplumber openai pillow networkx matplotlib arabic-reshaper python-bidi -q
import importlib, subprocess, sys

# Verifica installazione
for lib in ['pdfplumber','openai','PIL','networkx','matplotlib','arabic_reshaper','bidi']:
    try:
        importlib.import_module(lib)
        print(f"✓ {lib}")
    except:
        print(f"✗ {lib} — reinstallo...")
        subprocess.run([sys.executable, "-m", "pip", "install", lib, "-q"])

print("\n✓ Tutte le librerie OK")

# ============================================================
# CELLA 2 — Importazioni
# ============================================================
import pdfplumber
import json
import re
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import io
from openai import OpenAI
from google.colab import files
import arabic_reshaper
from bidi.algorithm import get_display

plt.rcParams['font.family'] = 'DejaVu Sans'
print("✓ Importazioni OK")

# ============================================================
# CELLA 3 — Configura OpenRouter
# ============================================================
API_KEY = "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"  # ← INCOLLA QUI LA TUA KEY

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=API_KEY,
    default_headers={
        "HTTP-Referer": "https://colab.research.google.com",
        "X-Title": "Progetto Arabo"
    }
)

def chiama_llm(prompt):
    modelli = [
        "google/gemma-3-4b-it:free",
        "meta-llama/llama-3.2-3b-instruct:free",
        "mistralai/mistral-7b-instruct:free",
    ]
    for modello in modelli:
        try:
            print(f"  Provo: {modello}...")
            risposta = client.chat.completions.create(
                model=modello,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=2000
            )
            print(f"  ✓ Funziona: {modello}")
            return risposta.choices[0].message.content
        except Exception as e:
            print(f"  ✗ {modello}: {e}")
            continue
    return None

test = chiama_llm("Rispondi solo con la parola: OK")
print(f"\nConnessione: {'✓ OK' if test else '✗ FALLITA'}")

# ============================================================
# CELLA 4 — Carica PDF arabo
# ============================================================
print("\nCarica il tuo PDF arabo:")
uploaded = files.upload()
nome_pdf = list(uploaded.keys())[0]
print(f"✓ Caricato: {nome_pdf}")

# ============================================================
# CELLA 5 — Leggi PDF e correggi RTL
# ============================================================
 # ============================================================
# CELLA 5 — Leggi PDF e correggi RTL (versione migliorata)
# ============================================================
import unicodedata

def correggi_arabo(testo):
    righe = testo.split('\n')
    righe_corrette = []
    for riga in righe:
        if riga.strip():
            try:
                # Inverti le parole (PDF arabo spesso le inverte)
                parole = riga.strip().split()
                parole_invertite = ' '.join(reversed(parole))
                # Applica reshaper e bidi
                reshaped = arabic_reshaper.reshape(parole_invertite)
                corretta = get_display(reshaped)
                righe_corrette.append(corretta)
            except:
                righe_corrette.append(riga)
        else:
            righe_corrette.append(riga)
    return '\n'.join(righe_corrette)

def correggi_label(nome):
    try:
        # Inverti parole + reshaper + bidi
        parole = nome.strip().split()
        parole_invertite = ' '.join(reversed(parole))
        reshaped = arabic_reshaper.reshape(parole_invertite)
        return get_display(reshaped)
    except:
        return nome

def leggi_pdf(percorso):
    testo_pagine = []
    with pdfplumber.open(percorso) as pdf:
        print(f"Pagine totali: {len(pdf.pages)}")
        for i, pagina in enumerate(pdf.pages):
            testo = pagina.extract_text()
            if testo and testo.strip():
                testo_pagine.append(testo.strip())
                print(f"  ✓ Pagina {i+1}: {len(testo)} caratteri")
            else:
                print(f"  ⚠ Pagina {i+1}: vuota o scansionata")
    testo_grezzo = "\n\n".join(testo_pagine)
    testo_rtl = correggi_arabo(testo_grezzo)
    return testo_grezzo, testo_rtl

testo_grezzo, testo_rtl = leggi_pdf(nome_pdf)
print(f"\n✓ Totale: {len(testo_grezzo)} caratteri")
print("\nTesto corretto RTL:")
print(testo_rtl)
 def correggi_arabo(testo):
    righe = testo.split('\n')
    righe_corrette = []
    for riga in righe:
        if riga.strip():
            try:
                reshaped = arabic_reshaper.reshape(riga)
                corretta = get_display(reshaped)
                righe_corrette.append(corretta)
            except:
                righe_corrette.append(riga)
        else:
            righe_corrette.append(riga)
    return '\n'.join(righe_corrette)

def correggi_label(nome):
    try:
        reshaped = arabic_reshaper.reshape(nome)
        return get_display(reshaped)
    except:
        return nome

def leggi_pdf(percorso):
    testo_pagine = []
    with pdfplumber.open(percorso) as pdf:
        print(f"Pagine totali: {len(pdf.pages)}")
        for i, pagina in enumerate(pdf.pages):
            testo = pagina.extract_text()
            if testo and testo.strip():
                testo_pagine.append(testo.strip())
                print(f"  ✓ Pagina {i+1}: {len(testo)} caratteri")
            else:
                print(f"  ⚠ Pagina {i+1}: vuota o scansionata")
    testo_grezzo = "\n\n".join(testo_pagine)
    testo_rtl = correggi_arabo(testo_grezzo)
    return testo_grezzo, testo_rtl

testo_grezzo, testo_rtl = leggi_pdf(nome_pdf)
print(f"\n✓ Totale: {len(testo_grezzo)} caratteri")
print("\nTesto corretto RTL:")
print(testo_rtl[:400])

# ============================================================
# CELLA 6 — Estrai concetti con LLM
# ============================================================
def estrai_concetti(testo, max_chars=3000):
    testo_troncato = testo[:max_chars]
    prompt = f"""
Analizza questo testo in arabo ed estrai concetti e relazioni.
Rispondi SOLO con JSON valido, senza testo aggiuntivo:
{{
  "concetti": [
    {{"id": "C1", "nome": "...", "categoria": "persona|luogo|idea|evento|oggetto"}}
  ],
  "relazioni": [
    {{"da": "C1", "a": "C2", "etichetta": "..."}}
  ]
}}

Testo:
{testo_troncato}
"""
    risposta = chiama_llm(prompt)
    if not risposta:
        return None

    match = re.search(r'\{.*\}', risposta, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except:
            print("⚠ JSON non valido, risposta grezza:")
            print(risposta[:300])
    return None

dati = estrai_concetti(testo_grezzo)
if dati:
    print(f"\n✓ {len(dati['concetti'])} concetti:")
    for c in dati['concetti']:
        print(f"  • [{c['categoria']}] {c['nome']}")
    print(f"✓ {len(dati['relazioni'])} relazioni")
else:
    print("✗ Estrazione fallita")

# ============================================================
# CELLA 7 — Crea GIF animata
# ============================================================
def crea_gif(dati, output="mappa_arabo.gif"):
    if not dati:
        print("✗ Nessun dato")
        return None

    colori_cat = {
        "persona":  "#4A90D9",
        "luogo":    "#27AE60",
        "idea":     "#8E44AD",
        "evento":   "#E67E22",
        "oggetto":  "#E74C3C",
    }

    G = nx.DiGraph()
    id_to_nome = {}
    colori_nodi = {}

    for c in dati["concetti"]:
        G.add_node(c["id"])
        id_to_nome[c["id"]] = correggi_label(c["nome"])
        colori_nodi[c["id"]] = colori_cat.get(c["categoria"], "#95A5A6")

    edge_labels = {}
    for r in dati["relazioni"]:
        if r["da"] in G.nodes and r["a"] in G.nodes:
            G.add_edge(r["da"], r["a"])
            edge_labels[(r["da"], r["a"])] = correggi_label(r["etichetta"])

    pos = nx.kamada_kawai_layout(G)
    nodi_lista = list(G.nodes())
    frames = []

    # Frame titolo
    fig, ax = plt.subplots(figsize=(12, 8))
    fig.patch.set_facecolor("#1a1a2e")
    ax.set_facecolor("#1a1a2e")
    titolo_ar = correggi_label("مخطط مفاهيمي")
    ax.text(0.5, 0.55, titolo_ar,
            ha='center', va='center', fontsize=28,
            color='white', fontweight='bold',
            transform=ax.transAxes)
    ax.text(0.5, 0.42, "Mappa Concettuale — Testo Arabo",
            ha='center', va='center', fontsize=14,
            color='#aaaaaa', transform=ax.transAxes)
    ax.axis('off')
    buf = io.BytesIO()
    plt.savefig(buf, format='png', facecolor="#1a1a2e", bbox_inches='tight')
    buf.seek(0)
    frames.append(Image.open(buf).copy())
    plt.close()

    # Frame per ogni nodo
    for k in range(1, len(nodi_lista) + 1):
        nodi_vis = nodi_lista[:k]
        sub = G.subgraph(nodi_vis)
        pos_sub = {n: pos[n] for n in nodi_vis}
        colori = [colori_nodi.get(n, "#95A5A6") for n in nodi_vis]
        labels = {n: id_to_nome.get(n, n) for n in nodi_vis}

        fig, ax = plt.subplots(figsize=(12, 8))
        fig.patch.set_facecolor("#1a1a2e")
        ax.set_facecolor("#1a1a2e")

        nx.draw_networkx_nodes(sub, pos_sub, node_color=colori,
                               node_size=3000, alpha=0.92, ax=ax)
        nx.draw_networkx_labels(sub, pos_sub, labels=labels,
                                font_size=10, font_color="white",
                                font_weight="bold", ax=ax)
        if k > 1:
            nx.draw_networkx_edges(sub, pos_sub,
                                   edge_color="#aaaaaa",
                                   arrows=True, arrowsize=15,
                                   width=1.5, ax=ax,
                                   connectionstyle="arc3,rad=0.1")
            el = {e: v for e, v in edge_labels.items()
                  if e[0] in nodi_vis and e[1] in nodi_vis}
            nx.draw_networkx_edge_labels(sub, pos_sub,
                                         edge_labels=el,
                                         font_color="#dddddd",
                                         font_size=8, ax=ax)

        legenda = [mpatches.Patch(color=v, label=k)
                   for k, v in colori_cat.items()]
        ax.legend(handles=legenda, loc="upper left",
                  fontsize=8, framealpha=0.3,
                  labelcolor='white',
                  facecolor='#2a2a3e')

        ax.set_title("Mappa Concettuale — Testo Arabo",
                     color='white', fontsize=13, pad=10)
        ax.axis('off')

        buf = io.BytesIO()
        plt.savefig(buf, format='png', facecolor="#1a1a2e", bbox_inches='tight')
        buf.seek(0)
        frames.append(Image.open(buf).copy())
        plt.close()
        print(f"  ✓ Frame {k}/{len(nodi_lista)}")

    # Pausa finale
    for _ in range(6):
        frames.append(frames[-1].copy())

    # Salva GIF
    frames[0].save(
        output,
        save_all=True,
        append_images=frames[1:],
        duration=900,
        loop=0
    )
    print(f"\n✓ GIF salvata: {output}")
    return output

gif = crea_gif(dati)

# ============================================================
# CELLA 8 — Mostra e scarica GIF
# ============================================================
from IPython.display import Image as IPImage, display

display(IPImage(filename=gif))
files.download(gif)
print("✓ GIF scaricata sul tuo PC!")

IndentationError: unexpected indent (712896872.py, line 141)

In [13]:
# ============================================================
# CELLA 1 — Installa tutto
# ============================================================
!pip install pdfplumber openai pillow networkx matplotlib arabic-reshaper python-bidi -q
print("✓ Librerie installate")

# ============================================================
# CELLA 2 — Importazioni
# ============================================================
import pdfplumber
import json
import re
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import io
from openai import OpenAI
from google.colab import files
import arabic_reshaper
from bidi.algorithm import get_display

plt.rcParams['font.family'] = 'DejaVu Sans'
print("✓ Importazioni OK")

# ============================================================
# CELLA 3 — Configura OpenRouter
# ============================================================
API_KEY = "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"  # ← INCOLLA QUI LA TUA KEY

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=API_KEY,
    default_headers={
        "HTTP-Referer": "https://colab.research.google.com",
        "X-Title": "Progetto Arabo"
    }
)

def chiama_llm(prompt):
    modelli = [
        "google/gemma-3-4b-it:free",
        "meta-llama/llama-3.2-3b-instruct:free",
        "mistralai/mistral-7b-instruct:free",
    ]
    for modello in modelli:
        try:
            print(f"  Provo: {modello}...")
            risposta = client.chat.completions.create(
                model=modello,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=2000
            )
            print(f"  ✓ Funziona: {modello}")
            return risposta.choices[0].message.content
        except Exception as e:
            print(f"  ✗ {modello}: {e}")
            continue
    return None

test = chiama_llm("Rispondi solo con la parola: OK")
print(f"\nConnessione: {'✓ OK' if test else '✗ FALLITA'}")

# ============================================================
# CELLA 4 — Carica PDF arabo
# ============================================================
print("\nCarica il tuo PDF arabo:")
uploaded = files.upload()
nome_pdf = list(uploaded.keys())[0]
print(f"✓ Caricato: {nome_pdf}")

# ============================================================
# CELLA 5 — Leggi PDF e correggi RTL
# ============================================================
def correggi_arabo(testo):
    """Corregge testo arabo invertito da PDF."""
    righe = testo.split('\n')
    righe_corrette = []
    for riga in righe:
        if riga.strip():
            try:
                # Inverti ordine parole (PDF arabo le inverte)
                parole = riga.strip().split()
                parole_invertite = ' '.join(reversed(parole))
                # Applica reshaper + bidi per RTL corretto
                reshaped = arabic_reshaper.reshape(parole_invertite)
                corretta = get_display(reshaped)
                righe_corrette.append(corretta)
            except:
                righe_corrette.append(riga)
        else:
            righe_corrette.append(riga)
    return '\n'.join(righe_corrette)

def correggi_label(nome):
    """Corregge singola parola/frase araba per visualizzazione."""
    try:
        parole = nome.strip().split()
        parole_invertite = ' '.join(reversed(parole))
        reshaped = arabic_reshaper.reshape(parole_invertite)
        return get_display(reshaped)
    except:
        return nome

def leggi_pdf(percorso):
    testo_pagine = []
    with pdfplumber.open(percorso) as pdf:
        print(f"Pagine totali: {len(pdf.pages)}")
        for i, pagina in enumerate(pdf.pages):
            testo = pagina.extract_text()
            if testo and testo.strip():
                testo_pagine.append(testo.strip())
                print(f"  ✓ Pagina {i+1}: {len(testo)} caratteri")
            else:
                print(f"  ⚠ Pagina {i+1}: vuota o scansionata")
    testo_grezzo = "\n\n".join(testo_pagine)
    testo_rtl = correggi_arabo(testo_grezzo)
    return testo_grezzo, testo_rtl

testo_grezzo, testo_rtl = leggi_pdf(nome_pdf)
print(f"\n✓ Totale: {len(testo_grezzo)} caratteri")
print("\n── Testo originale dal PDF ──")
print(testo_grezzo)
print("\n── Testo corretto RTL ──")
print(testo_rtl)

# ============================================================
# CELLA 6 — Estrai concetti con LLM
# ============================================================
def estrai_concetti(testo, max_chars=3000):
    testo_troncato = testo[:max_chars]
    prompt = f"""
Sei un esperto di lingua araba.
Analizza questo testo in arabo ed estrai i concetti principali e le relazioni tra loro.
I nomi dei concetti devono essere in arabo corretto.

Rispondi SOLO con JSON valido, senza testo aggiuntivo:
{{
  "concetti": [
    {{"id": "C1", "nome": "concetto in arabo", "categoria": "persona|luogo|idea|evento|oggetto"}}
  ],
  "relazioni": [
    {{"da": "C1", "a": "C2", "etichetta": "relazione in arabo"}}
  ]
}}

Testo arabo:
{testo_troncato}
"""
    risposta = chiama_llm(prompt)
    if not risposta:
        return None

    match = re.search(r'\{.*\}', risposta, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except:
            print("⚠ JSON non valido, risposta grezza:")
            print(risposta[:300])
    return None

dati = estrai_concetti(testo_grezzo)
if dati:
    print(f"\n✓ {len(dati['concetti'])} concetti trovati:")
    for c in dati['concetti']:
        nome_corretto = correggi_label(c['nome'])
        print(f"  • [{c['categoria']}] {c['nome']} → {nome_corretto}")
    print(f"✓ {len(dati['relazioni'])} relazioni trovate")
else:
    print("✗ Estrazione fallita")

# ============================================================
# CELLA 7 — Crea GIF animata
# ============================================================
def crea_gif(dati, output="mappa_arabo.gif"):
    if not dati:
        print("✗ Nessun dato")
        return None

    colori_cat = {
        "persona":  "#4A90D9",
        "luogo":    "#27AE60",
        "idea":     "#8E44AD",
        "evento":   "#E67E22",
        "oggetto":  "#E74C3C",
    }

    G = nx.DiGraph()
    id_to_nome = {}
    colori_nodi = {}

    for c in dati["concetti"]:
        G.add_node(c["id"])
        # Correggi RTL per ogni nodo
        id_to_nome[c["id"]] = correggi_label(c["nome"])
        colori_nodi[c["id"]] = colori_cat.get(c["categoria"], "#95A5A6")

    edge_labels = {}
    for r in dati["relazioni"]:
        if r["da"] in G.nodes and r["a"] in G.nodes:
            G.add_edge(r["da"], r["a"])
            edge_labels[(r["da"], r["a"])] = correggi_label(r["etichetta"])

    pos = nx.kamada_kawai_layout(G)
    nodi_lista = list(G.nodes())
    frames = []

    # ── Frame titolo iniziale ──
    fig, ax = plt.subplots(figsize=(12, 8))
    fig.patch.set_facecolor("#1a1a2e")
    ax.set_facecolor("#1a1a2e")
    titolo_ar = correggi_label("مخطط مفاهيمي")
    ax.text(0.5, 0.58, titolo_ar,
            ha='center', va='center', fontsize=30,
            color='white', fontweight='bold',
            transform=ax.transAxes)
    ax.text(0.5, 0.44, "Mappa Concettuale — Testo Arabo",
            ha='center', va='center', fontsize=15,
            color='#aaaaaa', transform=ax.transAxes)
    ax.axis('off')
    buf = io.BytesIO()
    plt.savefig(buf, format='png', facecolor="#1a1a2e", bbox_inches='tight')
    buf.seek(0)
    frames.append(Image.open(buf).copy())
    plt.close()

    # ── Frame per ogni nodo che appare ──
    for k in range(1, len(nodi_lista) + 1):
        nodi_vis = nodi_lista[:k]
        sub = G.subgraph(nodi_vis)
        pos_sub = {n: pos[n] for n in nodi_vis}
        colori = [colori_nodi.get(n, "#95A5A6") for n in nodi_vis]
        labels = {n: id_to_nome.get(n, n) for n in nodi_vis}

        fig, ax = plt.subplots(figsize=(12, 8))
        fig.patch.set_facecolor("#1a1a2e")
        ax.set_facecolor("#1a1a2e")

        nx.draw_networkx_nodes(sub, pos_sub,
                               node_color=colori,
                               node_size=3500,
                               alpha=0.92, ax=ax)
        nx.draw_networkx_labels(sub, pos_sub,
                                labels=labels,
                                font_size=10,
                                font_color="white",
                                font_weight="bold", ax=ax)
        if k > 1:
            nx.draw_networkx_edges(sub, pos_sub,
                                   edge_color="#aaaaaa",
                                   arrows=True,
                                   arrowsize=15,
                                   width=1.5, ax=ax,
                                   connectionstyle="arc3,rad=0.1")
            el = {e: v for e, v in edge_labels.items()
                  if e[0] in nodi_vis and e[1] in nodi_vis}
            nx.draw_networkx_edge_labels(sub, pos_sub,
                                         edge_labels=el,
                                         font_color="#dddddd",
                                         font_size=8, ax=ax)

        # Legenda categorie
        legenda = [mpatches.Patch(color=v, label=k)
                   for k, v in colori_cat.items()]
        ax.legend(handles=legenda,
                  loc="upper left",
                  fontsize=9,
                  framealpha=0.3,
                  labelcolor='white',
                  facecolor='#2a2a3e',
                  title="Categorie",
                  title_fontsize=9)

        ax.set_title("Mappa Concettuale — Testo Arabo",
                     color='white', fontsize=13, pad=10)
        ax.axis('off')

        buf = io.BytesIO()
        plt.savefig(buf, format='png',
                    facecolor="#1a1a2e",
                    bbox_inches='tight')
        buf.seek(0)
        frames.append(Image.open(buf).copy())
        plt.close()
        print(f"  ✓ Frame {k}/{len(nodi_lista)}")

    # ── Pausa finale (6 frame extra) ──
    for _ in range(6):
        frames.append(frames[-1].copy())

    # ── Salva GIF ──
    frames[0].save(
        output,
        save_all=True,
        append_images=frames[1:],
        duration=900,
        loop=0
    )
    print(f"\n✓ GIF salvata: {output}")
    return output

gif = crea_gif(dati)

# ============================================================
# CELLA 8 — Mostra e scarica GIF
# ============================================================
from IPython.display import Image as IPImage, display

if gif:
    display(IPImage(filename=gif))
    files.download(gif)
    print("✓ GIF scaricata sul tuo PC!")
else:
    print("✗ GIF non creata — controlla gli errori sopra")

✓ Librerie installate
✓ Importazioni OK
  Provo: google/gemma-3-4b-it:free...
  ✓ Funziona: google/gemma-3-4b-it:free

Connessione: ✓ OK

Carica il tuo PDF arabo:


Saving documento_arabo.pdf to documento_arabo (3).pdf
✓ Caricato: documento_arabo (3).pdf
Pagine totali: 1
  ✓ Pagina 1: 88 caratteri

✓ Totale: 88 caratteri

── Testo originale dal PDF ──
ضرلا ىلا ةقاطلا يطعت ثيح ةايحلا يف مهم رود بعلتو ةيسمشلا ةعومجملا يف ةمجن نع ةرابع سمشلا

── Testo corretto RTL ──
ﻻﺮﺿ ﻼﯨ ﻼﻃﺎﻗﺓ ﺖﻌﻄﻳ ﺢﻴﺛ ﻼﺤﻳﺍﺓ ﻒﻳ ﻢﻬﻣ ﺩﻭﺭ ﻮﺘﻠﻌﺑ ﻼﺸﻤﺴﻳﺓ ﻼﻤﺠﻣﻮﻋﺓ ﻒﻳ ﻦﺠﻣﺓ ﻊﻧ ﻊﺑﺍﺭﺓ ﻼﺸﻤﺳ
  Provo: google/gemma-3-4b-it:free...
  ✗ google/gemma-3-4b-it:free: Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'google/gemma-3-4b-it:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Google AI Studio', 'is_byok': False}}, 'user_id': 'user_3CUaDegN7kFXunlllTCNqtUc08t'}
  Provo: meta-llama/llama-3.2-3b-instruct:free...
  ✗ meta-llama/llama-3.2-3b-instruct:free: Error code: 429 - {'error': {'message': 'Provider returned erro